# Sesión de clase 06 · Ejercicio 2: clasificación de reseñas sin entrenamiento

**Clasificación zero-shot**: en vez de entrenar un clasificador con datos
etiquetados, definimos cada categoría como una frase descriptiva, la
vectorizamos igual que cualquier otro texto y asignamos a cada reseña la
etiqueta cuyo vector sea más parecido al suyo. No hace falta ni un solo
ejemplo etiquetado a mano.

Este notebook reproduce el **Ejercicio 2** de la clase sobre
`data/resenas_entrega.csv` (110 reseñas de entrega, con su calificación de
1 a 5 estrellas).

In [1]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

pd.set_option("display.max_colwidth", 100)

resenas = pd.read_csv("../data/resenas_entrega.csv")
print(resenas.shape)
resenas[["cliente_id", "producto_id", "calificacion", "texto"]].head()

(110, 7)


,cliente_id,producto_id,calificacion,texto
0,73,10,1,"Recibi el pedido incompleto, faltaba el cable de carga original del OnePlus 12 dentro de la caja..."
1,88,89,3,"El segundo cargador Belkin llego sin contratiempos, en el tiempo estimado y correctamente empacado."
2,28,45,5,La segunda Switch Lite que regale llego perfecta y justo a tiempo para el cumpleanos que la nece...
3,54,34,5,"El segundo television TCL llego perfecto, bien empacado y sin ningun dano, muy distinto a la pri..."
4,80,83,4,Me confirmaron por correo la garantia de un ano del cable Anker sin ningun problema.


## 1. Definir las etiquetas

Cinco categorías como frases descriptivas, no como palabras sueltas: el
embedding representa mejor una frase completa que una sola palabra.

In [2]:
etiquetas = {
    "retraso": "Retraso en la entrega",
    "danado": "Producto dañado o defectuoso",
    "incompleto": "Pedido incompleto o accesorio equivocado",
    "buena_entrega": "Entrega rápida y en buen estado",
    "soporte": "Atención de soporte o posventa",
}
etiquetas

{'retraso': 'Retraso en la entrega',
 'danado': 'Producto dañado o defectuoso',
 'incompleto': 'Pedido incompleto o accesorio equivocado',
 'buena_entrega': 'Entrega rápida y en buen estado',
 'soporte': 'Atención de soporte o posventa'}

## 2. Vectorizar etiquetas y reseñas

Guardamos las etiquetas como documentos de una colección de Chroma: así
reutilizamos `query()` para encontrar, para cada reseña, la etiqueta más
cercana (equivale a comparar la reseña contra cada una de las 5 frases y
quedarnos con la de mayor similitud coseno).

In [3]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

cliente = chromadb.Client()
col_etiquetas = cliente.get_or_create_collection(
    name="etiquetas_resenas",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

col_etiquetas.add(
    ids=list(etiquetas.keys()),
    documents=list(etiquetas.values()),
)
col_etiquetas.count()

/Users/erichuiza/Documents/pucp/miería web/2026-2/dev/sesion-de-clase-06/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25647.32it/s]

5

## 3. Clasificar cada reseña

Para cada texto de reseña, consultamos la colección de etiquetas con
`n_results=1`: gana la etiqueta más parecida. Chroma acepta una lista de
consultas en una sola llamada, así que clasificamos las 110 reseñas en un
solo `query()`.

In [4]:
res = col_etiquetas.query(query_texts=resenas["texto"].tolist(), n_results=1)

resenas = resenas.copy()
resenas["etiqueta_id"] = [ids[0] for ids in res["ids"]]
resenas["etiqueta"] = [docs[0] for docs in res["documents"]]
resenas["similitud"] = [round(1 - dists[0], 3) for dists in res["distances"]]

resenas[["calificacion", "etiqueta", "similitud", "texto"]].head(10)

,calificacion,etiqueta,similitud,texto
0,1,Pedido incompleto o accesorio equivocado,0.475,"Recibi el pedido incompleto, faltaba el cable de carga original del OnePlus 12 dentro de la caja..."
1,3,Entrega rápida y en buen estado,0.473,"El segundo cargador Belkin llego sin contratiempos, en el tiempo estimado y correctamente empacado."
2,5,Entrega rápida y en buen estado,0.288,La segunda Switch Lite que regale llego perfecta y justo a tiempo para el cumpleanos que la nece...
3,5,Entrega rápida y en buen estado,0.208,"El segundo television TCL llego perfecto, bien empacado y sin ningun dano, muy distinto a la pri..."
4,4,Retraso en la entrega,0.249,Me confirmaron por correo la garantia de un ano del cable Anker sin ningun problema.
5,5,Entrega rápida y en buen estado,0.173,"Los audifonos Bose se emparejaron al instante con mi telefono, cero complicaciones."
6,2,Retraso en la entrega,0.270,"La instalacion de este segundo pedido de Family Hub se retraso otra vez, casi diez dias despues ..."
7,3,Entrega rápida y en buen estado,0.455,"La segunda camara instantanea llego completa y a tiempo, sin ningun inconveniente en esta ocasion."
8,5,Entrega rápida y en buen estado,0.326,Cambiar el color de mi Marshall por otro modelo fue un proceso rapido y sin costo.
9,4,Entrega rápida y en buen estado,0.203,El Amazfit se sincronizo sin problemas con la app apenas lo encendi.


In [5]:
ejemplo = resenas.iloc[0]
print("Reseña: ", ejemplo["texto"])
print("Etiqueta asignada:", ejemplo["etiqueta"], f"(similitud {ejemplo['similitud']})")

Reseña:  Recibi el pedido incompleto, faltaba el cable de carga original del OnePlus 12 dentro de la caja. Tuve que esperar un envio adicional solo para recibir ese accesorio.
Etiqueta asignada: Pedido incompleto o accesorio equivocado (similitud 0.475)


## 4. Cruzar con la calificación (1–5★)

Si la clasificación captura bien el contenido de las reseñas, las
categorías de problemas (retraso, dañado, incompleto) deberían concentrar
las calificaciones bajas, y «entrega rápida y en buen estado» las
calificaciones altas.

In [6]:
tabla = pd.crosstab(resenas["etiqueta"], resenas["calificacion"])
tabla

calificacion,1,2,3,4,5
etiqueta,,,,,
Atención de soporte o posventa,0,0,0,1,0
Entrega rápida y en buen estado,0,4,9,16,26
Pedido incompleto o accesorio equivocado,8,4,9,3,1
Producto dañado o defectuoso,1,2,1,2,4
Retraso en la entrega,1,5,2,7,4


In [7]:
resumen = (
    resenas.groupby("etiqueta")["calificacion"]
    .agg(n="count", calificacion_promedio="mean")
    .sort_values("calificacion_promedio")
    .round(2)
)
resumen

,n,calificacion_promedio
etiqueta,,
Pedido incompleto o accesorio equivocado,25,2.40
Retraso en la entrega,19,3.42
Producto dañado o defectuoso,10,3.60
Atención de soporte o posventa,1,4.00
Entrega rápida y en buen estado,55,4.16


## Discusión

- ¿Qué categoría concentra las calificaciones de 1★ y 2★? Revisen la tabla
  cruzada: ¿coincide con lo esperado (retraso, dañado, incompleto)?
- ¿Cómo cambia el resultado si reformulan alguna etiqueta? Por ejemplo,
  cambien `"Producto dañado o defectuoso"` por `"El producto llegó roto"` y
  vuelvan a ejecutar la celda de clasificación: ¿algunas reseñas cambian de
  categoría? ¿la similitud promedio sube o baja?
- Esta clasificación no usó ni un solo ejemplo etiquetado a mano: solo la
  descripción de cada categoría. ¿En qué casos esto sería insuficiente y
  haría falta un clasificador entrenado con ejemplos reales?

In [26]:
# Espacio para experimentar: reformulen una etiqueta y comparen resultados
# Usamos upsert() (no add()): si vuelven a ejecutar esta celda tras cambiar
# el texto de una etiqueta, add() ignoraría el id repetido y dejaria el
# embedding viejo sin actualizar. upsert() sí reemplaza el documento y
# vuelve a vectorizarlo.
etiquetas_v2 = dict(etiquetas)
etiquetas_v2["danado"] = "El producto llegó roto"

col_etiquetas_v2 = cliente.get_or_create_collection(
    name="etiquetas_resenas_v2",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)
col_etiquetas_v2.upsert(ids=list(etiquetas_v2.keys()), documents=list(etiquetas_v2.values()))

res_v2 = col_etiquetas_v2.query(query_texts=resenas["texto"].tolist(), n_results=1)
etiqueta_v2 = pd.Series([docs[0] for docs in res_v2["documents"]], index=resenas.index)

cambiaron = resenas["etiqueta"] != etiqueta_v2
print(f"{cambiaron.sum()} de {len(resenas)} reseñas cambiaron de categoría al reformular la etiqueta")
resenas.loc[cambiaron, ["texto", "etiqueta"]].assign(etiqueta_v2=etiqueta_v2[cambiaron])

24 de 110 reseñas cambiaron de categoría al reformular la etiqueta


,texto,etiqueta,etiqueta_v2
11,La segunda laptop HP que compre llego con la pantalla algo desalineada del chasis. Funciona bien...,Producto dañado o defectuoso,El producto llegó roto
13,El Galaxy S24 Ultra ya venia con la mayoria del software actualizado desde fabrica.,Producto dañado o defectuoso,El producto llegó roto
16,El soporte tardo en responder sobre un problema de bateria del Moto G84.,Producto dañado o defectuoso,El producto llegó roto
22,El segundo Samsung The Frame que compre llego con un pixel muerto visible en la esquina superior...,Producto dañado o defectuoso,El producto llegó roto
28,El equipo de soporte reemplazo rapido un cable defectuoso de mis audifonos Bose.,Producto dañado o defectuoso,El producto llegó roto
29,El Galaxy Watch6 llego antes de la fecha limite que me habian prometido.,Entrega rápida y en buen estado,El producto llegó roto
33,El disco WD Blue llego bien protegido contra golpes con plastico de burbuja.,Producto dañado o defectuoso,El producto llegó roto
47,"El Garmin Venu 3 llego con la pantalla ya rayada, algo que no correspondia a un producto nuevo. ...",Pedido incompleto o accesorio equivocado,El producto llegó roto
50,La Lenovo IdeaPad llego con un rayon leve en la tapa superior que no se veia en las fotos del pr...,Entrega rápida y en buen estado,El producto llegó roto
51,"El Galaxy A55 llego en la fecha estimada, la caja se veia algo usada pero el telefono estaba per...",Pedido incompleto o accesorio equivocado,El producto llegó roto
